# Embedding Model Evaluation

#### Compare three embedding models on the Baldwin Boro regulation PDFs

| Model | Type | Cost |
|---|---|---|
| `all-MiniLM-L6-v2` | ChromaDB default (local) | Free |
| `all-mpnet-base-v2` | HuggingFace sentence-transformers (local) | Free |
| `text-embedding-3-small` | OpenAI API | ~$0.02 / 1M tokens |

Each model gets its own ChromaDB collection. The same queries are run against all three and results are printed side-by-side.


In [1]:
import os
import re
import chromadb
import pypdf
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from chromadb.utils.embedding_functions import (
    DefaultEmbeddingFunction,
    SentenceTransformerEmbeddingFunction,
    OpenAIEmbeddingFunction,
)

load_dotenv(find_dotenv())

SECTION_PATTERN = re.compile(
    r'(?=(?:Chapter\s+[A-Z]?\d+|ARTICLE\s+[IVXLC]+|§\s*[A-Z]?\d+[-\w]*\.))',
    re.MULTILINE
)
OVERLAP_CHARS = 200

def extract_sections(pdf_path: Path) -> list[dict]:
    reader = pypdf.PdfReader(pdf_path)
    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    raw_chunks = SECTION_PATTERN.split(full_text)
    raw_chunks = [c.strip() for c in raw_chunks if c.strip()]

    sections = []
    for i, chunk in enumerate(raw_chunks):
        overlap = raw_chunks[i - 1][-OVERLAP_CHARS:] if i > 0 else ""
        heading = chunk.splitlines()[0].strip()
        sections.append({
            "text": overlap + chunk,
            "source": pdf_path.name,
            "chunk_index": i,
            "heading": heading,
        })
    return sections

In [2]:
data_dir = Path("data")
all_sections = []
for pdf_path in sorted(data_dir.glob("*.pdf")):
    all_sections.extend(extract_sections(pdf_path))

print(f"Loaded {len(all_sections)} chunks from {len(list(data_dir.glob('*.pdf')))} PDFs")

Loaded 2061 chunks from 3 PDFs


In [3]:
MODELS = {
    "minilm (default)": DefaultEmbeddingFunction(),
    "mpnet (huggingface)": SentenceTransformerEmbeddingFunction(model_name="all-mpnet-base-v2"),
    "text-embedding-3-small (openai)": OpenAIEmbeddingFunction(
        api_key=os.environ["OPENAI_API_KEY"],
        model_name="text-embedding-3-small",
    ),
}

client = chromadb.EphemeralClient()
collections = {}

for model_name, ef in MODELS.items():
    col = client.get_or_create_collection(
        name=model_name.replace(" ", "_").replace("(", "").replace(")", ""),
        embedding_function=ef,
    )
    col.add(
        documents=[s["text"] for s in all_sections],
        metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"], "heading": s["heading"]} for s in all_sections],
        ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
    )
    collections[model_name] = col
    print(f"Built collection: {model_name}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Built collection: minilm (default)
Built collection: mpnet (huggingface)


BadRequestError: Error code: 400 - {'error': {'message': 'Requested 395625 tokens, max 300000 tokens per request', 'type': 'max_tokens_per_request', 'param': None, 'code': 'max_tokens_per_request'}} in add.

In [ ]:
def compare(query: str, n_results: int = 3):
    print(f"\nQUERY: {query}\n{'=' * 80}")
    for model_name, col in collections.items():
        results = col.query(query_texts=[query], n_results=n_results)
        print(f"\n--- {model_name.upper()} ---")
        for text, meta, distance in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            print(f"  Score: {1 - distance:.3f} | {meta['source']} — {meta['heading']}")

compare("do i have to register with the boro if i want to install an alarm?")
compare("what are the rules for parking on residential streets?")
compare("how do i appeal a zoning decision?")